# Sprint 1 - Web scraping and property locations

This notebook establishes the Sprint 1 pipeline for MAST30034 Project 2. The current project decision is to plan a three-year rental-price forecast. This notebook does not claim to forecast future prices yet: it validates the listing schema, parser, and location map first.

The local HTML file is a synthetic fixture used only for testing. Live collection must use an authorised source, course skeleton, API, or export.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in candidates if (path / 'src' / 'scraper.py').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.scraper import extract_listing_records, records_to_frame
from src.visualisation import create_property_map

FORECAST_HORIZON_YEARS = 3
FIXTURE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'example_listing_page.html'
print(PROJECT_ROOT)
print(f'Forecast horizon: {FORECAST_HORIZON_YEARS} years')

## 1. Define the property-level schema

The target for the later rental model is `weekly_rent_aud`. The remaining fields are candidate internal features or identifiers for joining external data.

In [ ]:
schema = {
    'weekly_rent_aud': 'weekly asking rent in AUD',
    'bedrooms': 'number of bedrooms',
    'bathrooms': 'number of bathrooms',
    'parking_spaces': 'number of parking spaces',
    'property_type': 'house, apartment, townhouse, etc.',
    'suburb': 'suburb used for SA2 and business aggregation',
    'latitude': 'property latitude where provided',
    'longitude': 'property longitude where provided',
}
pd.Series(schema, name='definition').to_frame()

## 2. Test the parser on the local fixture

In [ ]:
html = FIXTURE_PATH.read_text(encoding='utf-8')
records = extract_listing_records(html, source_page_url=FIXTURE_PATH.as_uri())
listings = records_to_frame(records)
listings

In [ ]:
assert len(listings) == 3
assert listings['weekly_rent_aud'].notna().all()
assert listings['listing_id'].is_unique
assert listings['state'].dropna().eq('VIC').all()
print('Parser smoke test passed:', len(listings), 'synthetic records')

## 3. Visualise listing locations

The map is an early Sprint 1 deliverable. It will be replaced with the group's authorised Victorian listing sample when the real source is confirmed.

In [ ]:
map_path = PROJECT_ROOT / 'data' / 'processed' / 'sprint1_example_property_map.html'
property_map = create_property_map(listings, map_path)
print(f'Saved example map to {map_path}')
property_map

## 4. Live collection: keep opt-in

Fill `AUTHORISED_SOURCE_URLS` only after the group has confirmed that automated access is allowed. `crawl_pages` checks `robots.txt`, uses a descriptive user agent, spaces requests, and refuses disallowed pages. Never bypass access controls or rate limits.

In [ ]:
# Keep empty until an authorised source has been confirmed.
AUTHORISED_SOURCE_URLS = []

if AUTHORISED_SOURCE_URLS:
    from src.scraper import crawl_pages, save_records
    live_records = crawl_pages(AUTHORISED_SOURCE_URLS, min_delay_seconds=2.0)
    live_listings = save_records(live_records, PROJECT_ROOT / 'data' / 'processed' / 'sprint1_listings.csv')
    live_listings.head()
else:
    print('Live collection is disabled: confirm an authorised source before adding URLs.')